# Module SC8 — **Detection** (YOLO11) + end-to-end detect→classify

> **Run on Kaggle** — Settings: Accelerator **GPU T4 x2**, Internet **ON**, Environment **Always use latest**. Attach inputs listed below, then **Save Version (Save & Run All)** so the outputs can be chained into the next module.

Builds a single-class (`waste_item`) YOLO dataset from the manifests (boxes are already normalized → converted to YOLO center format), trains **YOLO11**, reports detection mAP/P/R, then runs the detector→crop→4-head classifier to compare **end-to-end** vs **oracle-crop** accuracy.

**Inputs:** `thesis-common-10c`, the image dataset, and (for end-to-end) the trainer outputs + SC6 output — it auto-picks the **best material backbone** from `best_per_head.json` (falls back to any attached `weights_<bb>.pt`).

**Outputs:** `detection_metrics.json`, `end_to_end.json`, trained `det/weights/best.pt`.

In [ ]:

import sys, glob, os, json
import numpy as np
# Provenance bootstrap: explicit content identity; no timestamp/size selection.
import hashlib
from pathlib import Path
_PINNED_HELPER_SHA256 = "57b5c16cc9a08c7a3b2afa02c90ece96859b207d4533c78657f862752b68c34b"
ARTIFACT_OVERRIDES = {}  # T.find_one-only paths; generic cached artifact conflicts still require detaching.
_local_input_root = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
_input_roots = [Path('/kaggle/input')] if Path('/kaggle/input').is_dir() else [_local_input_root]
def _content_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
def _input_candidates(pattern):
    hits = set()
    for root in _input_roots:
        for path in root.rglob(pattern):
            if path.is_file() and not {'Archives', '.git', 'node_modules'}.intersection(path.parts):
                hits.add(str(path.resolve()))
    return sorted(hits)
_helpers = _input_candidates('thesis_common.py')
_matching_helpers = [p for p in _helpers if _content_sha256(p) == _PINNED_HELPER_SHA256]
assert _matching_helpers, ('Attach the thesis_common.py supplied in Notebooks. '
                           'No input helper matches SHA256 ' + _PINNED_HELPER_SHA256)
_helper_path = _matching_helpers[0]
sys.path.insert(0, str(Path(_helper_path).parent))
import importlib, thesis_common as T
importlib.reload(T)
assert _content_sha256(T.__file__) == _PINNED_HELPER_SHA256, 'Wrong thesis_common imported.'
def _strict_find_one(filename_glob, required=True):
    if filename_glob in ARTIFACT_OVERRIDES:
        path = Path(ARTIFACT_OVERRIDES[filename_glob])
        assert path.is_file(), 'ARTIFACT_OVERRIDES path missing: ' + str(path)
        return str(path)
    hits = _input_candidates(filename_glob)
    if not hits:
        if required:
            raise FileNotFoundError('Attach the exact producer for ' + filename_glob + '; see the notebook preamble.')
        return None
    if len(hits) > 1:
        digests = {_content_sha256(p) for p in hits}
        if len(digests) != 1:
            raise RuntimeError('Different artifacts share ' + filename_glob + ':\n' + '\n'.join(hits)
                               + '\nDetach stale producers or set ARTIFACT_OVERRIDES explicitly.')
    return hits[0]
T.find_one = _strict_find_one
# Original local find()/find_all() consumers cannot obey T.find_one overrides.
# Reject conflicting standard cached artifacts up front instead of letting those
# consumers silently choose a same-name SC7/SC26 or stale output.
_generic_patterns = ('logits_*.npz', 'metrics_*.json', 'history_*.csv', 'calibration.json', 'config.json')
_by_basename = {}
for _pattern in _generic_patterns:
    for _p in _input_candidates(_pattern):
        _by_basename.setdefault(Path(_p).name, []).append(_p)
for _name, _paths in _by_basename.items():
    if len(_paths) > 1 and len({_content_sha256(p) for p in _paths}) > 1:
        raise RuntimeError('Conflicting input producers for ' + _name + ':\n' + '\n'.join(_paths)
                           + '\nAttach one producer/version per basename for this run.')
print('Pinned helper:', _helper_path, '| SHA256:', _PINNED_HELPER_SHA256)
_cfg = T.find_one('config.json', required=False)
if _cfg: T.load_config(_cfg); print('loaded config:', _cfg)
print('device:', T.DEVICE, '| materials:', len(T.MATERIALS), '| epochs:', T.EPOCHS)


In [ ]:

import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics'], check=False)
ROOT = T.find_image_root()
DATA_YAML = T.export_yolo_dataset(ROOT, '/kaggle/working/yolo_ds', include_aug=False)
print('data.yaml ->', DATA_YAML)

### Train YOLO11 (best small detector; bump to yolo11s/m for more accuracy)

In [ ]:

from ultralytics import YOLO
det = YOLO('yolo11s.pt')                       # 'yolo11n' faster, 'yolo11m' more accurate
det.train(data=DATA_YAML, epochs=50, imgsz=640, batch=16,
          project='/kaggle/working', name='det', exist_ok=True, verbose=False)

### Detection metrics on the test split (mAP / precision / recall)

In [ ]:

mt = det.val(data=DATA_YAML, split='test')
dm = {'map50': float(mt.box.map50), 'map50_95': float(mt.box.map),
      'precision': float(mt.box.mp), 'recall': float(mt.box.mr)}
json.dump(dm, open('detection_metrics.json','w'), indent=2)
print('DETECTION:', {k:round(v,4) for k,v in dm.items()})

### End-to-end: detector crop → classifier, vs oracle (ground-truth-box) crop

In [ ]:

# load a trained classifier if its weights are attached — PREFER the best material backbone (from SC6)
import torch, glob as _g
best_mat = None
_bph = T.find_one('best_per_head.json', required=False)
if _bph: best_mat = json.load(open(_bph)).get('material')
wpath = None
for nm in ([best_mat] if best_mat else []) + T.BACKBONES:
    if not nm: continue
    p = T.find_one(f'weights_{nm}.pt', required=False)
    if p: wpath, wbb = p, nm; break
if wpath is None:
    print('No classifier weights attached -> skipping end-to-end. (Attach a Module 2-5 weights_<bb>.pt.)')
else:
    print('end-to-end classifier:', wbb)
    clf = T.MultiTaskNet(wbb).to(T.DEVICE)
    clf.load_state_dict(torch.load(wpath, map_location=T.DEVICE)); clf.eval()
    import pandas as pd, numpy as np
    df = pd.read_csv(os.path.join(ROOT,'test.csv'))
    if 'source' in df.columns: df = df[df['source'].fillna('original')=='original']
    bidx = T._basename_index(ROOT,'test')
    def classify(img):
        im = T.gray_world(img); x = T.apply_tf(T.eval_tf, im)[None].to(T.DEVICE)
        with torch.no_grad(): o = clf(x)
        return int(o['material'].argmax(1))
    n=correct_oracle=correct_e2e=detected=0
    for _,r in df.iterrows():
        ap = T._resolve(ROOT,'test',r['item_id'],r['image_path'],bidx)
        if not ap: continue
        img = T.load_rgb(ap); y = T.MATERIALS.index(r['material']); n+=1
        # oracle crop (annotated box)
        oc = T.object_crop(img,(r.bbox_x,r.bbox_y,r.bbox_w,r.bbox_h))
        correct_oracle += int(classify(oc)==y)
        # detector crop (top box)
        res = det.predict(ap, verbose=False, conf=0.25)
        b = res[0].boxes
        if b is not None and len(b)>0:
            detected+=1
            xyxy = b.xyxy[int(b.conf.argmax())].tolist()
            dc = T.crop_from_pred(img, xyxy)
        else:
            dc = T.object_crop(img, (None,None,None,None))   # fallback center crop
        correct_e2e += int(classify(dc)==y)
    ee = {'n':n,'detect_rate':detected/max(n,1),
          'material_acc_oracle_crop':correct_oracle/max(n,1),
          'material_acc_end_to_end':correct_e2e/max(n,1)}
    json.dump(ee, open('end_to_end.json','w'), indent=2)
    print('END-TO-END:', {k:(round(v,4) if isinstance(v,float) else v) for k,v in ee.items()})